In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

In [2]:
os.getcwd()

'C:\\Users\\Tom\\Documents\\Project working directory\\Coursework\\Python files'

In [3]:
os.chdir('~\\Project working directory\\Coursework\\Python files')

FileNotFoundError: [WinError 3] The system cannot find the path specified: '~\\Project working directory\\Coursework\\Python files'

# Creating Database

In [4]:
try:
    os.remove('../Python files/py_airline4.db')
except OSError:
    pass

conn = sqlite3.connect('../Python files/py_+airline4.db')

# Creating table using airline data from 2001 to 2002

In [5]:
planes = pd.read_csv("../dataverse_files/plane-data.csv")
planes.to_sql('planes', con = conn, index = False)

In [6]:
c = conn.cursor()

In [7]:
c.execute('''
CREATE TABLE arrived (
  Year int,
  Month int,
  DayofMonth int,
  DayOfWeek int,
  DepTime  int,
  CRSDepTime int,
  ArrTime int,
  CRSArrTime int,
  UniqueCarrier varchar(5),
  FlightNum int,
  TailNum varchar(8),
  ActualElapsedTime int,
  CRSElapsedTime int,
  AirTime int,
  ArrDelay int,
  DepDelay int,
  Origin varchar(3),
  Dest varchar(3),
  Distance int,
  TaxiIn int,
  TaxiOut int,
  Cancelled int,
  CancellationCode varchar(1),
  Diverted varchar(1),
  CarrierDelay int,
  WeatherDelay int,
  NASDelay int,
  SecurityDelay int,
  LateAircraftDelay int
)
''')

conn.commit()

In [8]:
for year in range(2001, 2002):
    print('Processing: ', year)
    arrived1 = pd.read_csv("../Python files/"+str(year)+".csv.bz2", encoding='latin-1')    
    arrived1.to_sql('arrived1', con = conn, if_exists = 'append', index = False)
for year in range(2002, 2003):
    print('Processing: ', year)
    arrived2 = pd.read_csv("../Python files/"+str(year)+".csv.bz2", encoding='latin-1')    
    arrived2.to_sql('arrived2', con = conn, if_exists = 'append', index = False)
arrived = pd.concat([arrived1, arrived2])
conn.commit()

Processing:  2001
Processing:  2002


# Create dataframe

In [19]:
df1 = c.execute('''
 SELECT TailNum, Year, Month, DayofMonth, DayOfWeek, DepTime, CRSDepTime, ArrTime, CRSArrTime,
 Origin, Dest, ActualElapsedTime, CRSElapsedTime, FlightNum, ArrDelay, DepDelay
 FROM arrived1
 WHERE ArrDelay >0 AND DepDelay>0
 GROUP BY TailNum
 ORDER BY TailNum ASC
             ''').fetchall()
df2 = pd.DataFrame(df1)
df2.columns = ['TailNum', 'Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime',
 'Origin', 'Dest', 'ActualElapsedTime', 'CRSElapsedTime', 'FlightNum', 'ArrDelay', 'DepDelay']
df2

,TailNum,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,Origin,Dest,ActualElapsedTime,CRSElapsedTime,FlightNum,ArrDelay,DepDelay
0,-N037M,2001,1,18,4,1306.0,1255,1400.0,1345,EYW,MIA,54.0,50,5212,15.0,11.0
1,-N047M,2001,1,3,3,1056.0,1045,1153.0,1133,MIA,APF,57.0,48,5201,20.0,11.0
2,-N107D,2001,1,1,1,2221.0,1720,50.0,1940,LAS,SLC,89.0,80,670,310.0,301.0
3,-N108D,2001,1,24,3,533.0,530,728.0,718,FLL,ATL,115.0,108,624,10.0,3.0
4,-N109D,2001,1,5,5,538.0,530,727.0,718,FLL,ATL,109.0,108,624,9.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4551,NR78A1,2001,1,5,5,1020.0,1015,1203.0,1144,LAS,SJC,103.0,89,129,19.0,5.0
4552,NR79A1,2001,1,10,3,1017.0,1015,1207.0,1144,LAS,SJC,110.0,89,129,23.0,2.0
4553,NR80A1,2001,1,7,7,1102.0,1015,1229.0,1144,LAS,SJC,87.0,89,129,45.0,47.0
4554,NR81A1,2001,1,1,1,1207.0,1135,1325.0,1257,SJC,LAS,78.0,82,128,28.0,32.0


In [11]:
#Find Timing with most delays
df2.TailNum.value_counts().idxmax()

'NR82A1'

In [29]:
#Create a table to visualise flights in a day
df3 = df2.loc[df2['Month'] == 1]
df4 = df3.loc[df3['DayofMonth'] == 1]
df5 = df4.loc[df4['DayOfWeek'] == 1] 

df6 = df5.sort_values(by=['FlightNum'],)
df6.head(n=60)

,TailNum,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,Origin,Dest,ActualElapsedTime,CRSElapsedTime,FlightNum,ArrDelay,DepDelay
1833,N373A1,2001,1,1,1,1000.0,900,1242.0,1211,JFK,LAX,342.0,371,1,31.0,60.0
751,N13088,2001,1,1,1,941.0,940,1358.0,1355,IAH,HNL,497.0,495,1,3.0,1.0
1911,N385A1,2001,1,1,1,1205.0,1200,2028.0,2022,LAX,JFK,323.0,322,4,6.0,5.0
4395,N951ä@,2001,1,1,1,1553.0,1535,1834.0,1830,MIA,JFK,161.0,175,4,4.0,18.0
692,N10834,2001,1,1,1,1726.0,1720,1823.0,1816,IAH,SAT,57.0,56,5,7.0,6.0
4466,N972ãæ,2001,1,1,1,2127.0,2030,1.0,2317,MIA,JFK,154.0,167,6,44.0,57.0
3473,N725ãæ,2001,1,1,1,2215.0,2100,51.0,2352,SJU,JFK,216.0,232,9,59.0,75.0
1551,N322A1,2001,1,1,1,2202.0,2200,633.0,617,LAX,JFK,331.0,317,10,16.0,2.0
3434,N718ãæ,2001,1,1,1,1219.0,1210,1745.0,1737,SNA,STL,206.0,207,10,8.0,9.0
1541,N320A1,2001,1,1,1,1304.0,1300,2157.0,2123,LAX,BOS,353.0,323,12,34.0,4.0


In [ ]:
conn.close()